In [3]:
import os, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn as dglnn
from dgl.dataloading import GraphDataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score, classification_report

ROOT = Path.home() / "Desktop" / "\u671f\u672b\u8bba\u6587"
DATA_DIR = ROOT / "\u6a21\u578b\u642d\u5efa" / "data" / "\u6570\u636e\u96c6\u5212\u5206" / "\u7f51\u7edc\u6570\u636e"
SAVE_DIR = ROOT / "\u6a21\u578b\u642d\u5efa" / "data" / "\u4fdd\u5b58\u6a21\u578b" / "GNN"
BATCH_SIZE = 1
EPOCHS = 50
PATIENCE = 8
LR = 8e-4
WEIGHT_DECAY = 5e-4

CONT_NAMES = [
    'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE',
    'FLIGHTS', 'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD',
    'CRS_DEP_TIME', 'CRS_ARR_TIME',
]
CAT_NAMES = ['ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK', 'CRS_DEP_TIME_HOUR', 'CRS_ARR_TIME_HOUR']
EMB_DIMS = {
    'ORIGIN': 16,
    'DEST': 16,
    'MONTH': 4,
    'DAY_OF_WEEK': 4,
    'CRS_DEP_TIME_HOUR': 6,
    'CRS_ARR_TIME_HOUR': 6,
}

device = torch.device('cpu')
torch.backends.cudnn.benchmark = False
print('device:', device)

with open(DATA_DIR / 'network_info.json', encoding='utf-8') as f:
    info = json.load(f)
LABEL_THRESHOLD = info.get('label_threshold_minutes', 15)


def load_split(name):
    graphs, _ = dgl.load_graphs(str(DATA_DIR / f'network_{name}.dgl'))
    if not graphs:
        raise ValueError(f'network_{name}.dgl contains no graphs')
    return graphs

train_graphs = load_split('train')
val_graphs = load_split('val')
test_graphs = load_split('test')
print('graphs:', len(train_graphs), len(val_graphs), len(test_graphs))


def safe_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('wb') as f:
        torch.save(obj, f, _use_new_zipfile_serialization=False)


def safe_torch_load(path, **kwargs):
    with Path(path).open('rb') as f:
        return torch.load(f, **kwargs)


def node_col(g, name, dtype=torch.float32):
    return torch.nan_to_num(g.ndata[name].view(-1).to(dtype), nan=0.0, posinf=0.0, neginf=0.0)


def build_cont(g):
    return torch.stack([node_col(g, name) for name in CONT_NAMES], dim=1)


def build_cat(g):
    cols = []
    for name in CAT_NAMES:
        x = node_col(g, name, dtype=torch.float32).round().long()
        if name == 'MONTH':
            x = x.clamp(0, 12)
        elif name == 'DAY_OF_WEEK':
            x = x.clamp(0, 7)
        elif name.endswith('_HOUR'):
            x = x.clamp(0, 23)
        else:
            x = x.clamp_min(0)
        cols.append(x)
    return torch.stack(cols, dim=1)


def dataset_stats(graphs):
    total = 0
    cont_dim = len(CONT_NAMES)
    sum_x = torch.zeros(cont_dim, dtype=torch.float64)
    sum_x2 = torch.zeros(cont_dim, dtype=torch.float64)
    cat_max = {name: 0 for name in CAT_NAMES}
    pos = 0
    label_total = 0
    for g in graphs:
        x = build_cont(g).double()
        total += x.shape[0]
        sum_x += x.sum(0)
        sum_x2 += (x * x).sum(0)
        cats = build_cat(g)
        for i, name in enumerate(CAT_NAMES):
            cat_max[name] = max(cat_max[name], int(cats[:, i].max().item()))
        y = g.ndata['label_bin'].float().view(-1)
        pos += int(y.sum().item())
        label_total += y.numel()
    mean = (sum_x / total).float()
    var = (sum_x2 / total - mean.double() * mean.double()).clamp_min(1e-12).float()
    std = torch.sqrt(var)
    neg = label_total - pos
    pos_weight = neg / max(pos, 1)
    cat_sizes = {name: cat_max[name] + 1 for name in CAT_NAMES}
    return mean, std, cat_sizes, pos_weight, pos, label_total

cont_mean, cont_std, cat_sizes, pos_weight_value, pos_count, total_count = dataset_stats(train_graphs)
print(f'train positive rate: {pos_count / total_count:.4f}, pos_weight: {pos_weight_value:.4f}')
print('cat_sizes:', cat_sizes)


def prepare_graphs(graphs, mean, std):
    prepared = []
    for g in graphs:
        g = g.local_var()
        g.ndata['x_cont'] = (build_cont(g) - mean) / std
        g.ndata['x_cat'] = build_cat(g)
        g.ndata['label_bin'] = g.ndata['label_bin'].float().view(-1, 1)
        for key in list(g.edata.keys()):
            del g.edata[key]
        g = dgl.remove_self_loop(g)
        g = dgl.to_bidirected(g, copy_ndata=True)
        g = dgl.add_self_loop(g)
        prepared.append(g)
    return prepared

train_graphs = prepare_graphs(train_graphs, cont_mean, cont_std)
val_graphs = prepare_graphs(val_graphs, cont_mean, cont_std)
test_graphs = prepare_graphs(test_graphs, cont_mean, cont_std)

train_loader = GraphDataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = GraphDataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = GraphDataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)


class EmbeddedResidualSAGE(nn.Module):
    def __init__(self, cont_dim, cat_sizes, hidden=128, n_layers=2, drop=0.25):
        super().__init__()
        self.cat_names = CAT_NAMES
        self.embs = nn.ModuleDict({
            name: nn.Embedding(cat_sizes[name], EMB_DIMS[name])
            for name in self.cat_names
        })
        emb_dim = sum(EMB_DIMS[name] for name in self.cat_names)
        self.input_proj = nn.Sequential(
            nn.Linear(cont_dim + emb_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(drop),
        )
        self.layers = nn.ModuleList([
            dglnn.SAGEConv(hidden, hidden, aggregator_type='mean')
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(drop)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden, 1),
        )

    def encode(self, cont, cat):
        emb_parts = []
        for i, name in enumerate(self.cat_names):
            idx = cat[:, i].clamp(0, self.embs[name].num_embeddings - 1)
            emb_parts.append(self.embs[name](idx))
        return torch.cat([cont] + emb_parts, dim=1)

    def forward(self, g, cont, cat):
        h0 = self.input_proj(self.encode(cont, cat))
        h = h0
        for conv, norm in zip(self.layers, self.norms):
            msg = conv(g, h)
            h = norm(h + self.drop(F.gelu(msg)))
        return self.head(torch.cat([h, h0], dim=1)).squeeze(-1)


def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_nodes = 0
    for bg in loader:
        bg = bg.to(device)
        label = bg.ndata['label_bin'].view(-1)
        with torch.set_grad_enabled(train):
            logits = model(bg, bg.ndata['x_cont'], bg.ndata['x_cat'])
            loss = criterion(logits, label)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * label.numel()
        total_nodes += label.numel()
    return total_loss / max(total_nodes, 1)


def predict(loader):
    model.eval()
    probs, trues = [], []
    with torch.no_grad():
        for bg in loader:
            bg = bg.to(device)
            logits = model(bg, bg.ndata['x_cont'], bg.ndata['x_cat'])
            probs.append(torch.sigmoid(logits).detach().cpu())
            trues.append(bg.ndata['label_bin'].view(-1).detach().cpu())
    return torch.cat(probs).numpy(), torch.cat(trues).numpy().astype(int)


def best_thresholds(probs, trues):
    best_f1 = {'threshold': 0.5, 'f1': -1.0, 'acc': 0.0, 'bal_acc': 0.0}
    best_bal = {'threshold': 0.5, 'f1': 0.0, 'acc': 0.0, 'bal_acc': -1.0}
    for t in np.linspace(0.05, 0.95, 91):
        pred = (probs >= t).astype(int)
        item = {
            'threshold': float(t),
            'f1': float(f1_score(trues, pred, zero_division=0)),
            'acc': float(accuracy_score(trues, pred)),
            'bal_acc': float(balanced_accuracy_score(trues, pred)),
        }
        if item['f1'] > best_f1['f1']:
            best_f1 = item
        if item['bal_acc'] > best_bal['bal_acc']:
            best_bal = item
    return {'f1': best_f1, 'balanced': best_bal}


model = EmbeddedResidualSAGE(len(CONT_NAMES), cat_sizes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value ** 0.5, dtype=torch.float32, device=device))

os.makedirs(SAVE_DIR, exist_ok=True)
best_val = float('inf')
wait = 0
best_model_path = SAVE_DIR / 'gcn.pth'

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"Epoch {epoch + 1:2d} | loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        wait = 0
        safe_torch_save(model.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}")
            break

model.load_state_dict(safe_torch_load(best_model_path, map_location=device))
val_prob, val_true = predict(val_loader)
threshold_info = best_thresholds(val_prob, val_true)
print('best thresholds:', threshold_info)

test_prob, test_true = predict(test_loader)
auc = float(roc_auc_score(test_true, test_prob))
reports = {}
for key, info_t in threshold_info.items():
    pred = (test_prob >= info_t['threshold']).astype(int)
    reports[key] = {
        'acc': float(accuracy_score(test_true, pred)),
        'balanced_acc': float(balanced_accuracy_score(test_true, pred)),
        'auc': auc,
        'f1': float(f1_score(test_true, pred, zero_division=0)),
        'threshold': info_t['threshold'],
        'val_f1': info_t['f1'],
        'val_bal_acc': info_t['bal_acc'],
    }
    print(f"\n[{key}] threshold={info_t['threshold']:.2f}")
    print(f"ACC: {reports[key]['acc']:.4f}")
    print(f"BAL_ACC: {reports[key]['balanced_acc']:.4f}")
    print(f"AUC: {auc:.4f}")
    print(f"F1:  {reports[key]['f1']:.4f}")
    print(classification_report(test_true, pred, target_names=['normal', 'delay'], zero_division=0))

metrics = {
    'model': 'EmbeddedResidualSAGE',
    'continuous_features': CONT_NAMES,
    'categorical_features': CAT_NAMES,
    'cat_sizes': cat_sizes,
    'reports': reports,
    'pos_weight_raw': float(pos_weight_value),
    'pos_weight_used': float(pos_weight_value ** 0.5),
    'label_threshold_minutes': LABEL_THRESHOLD,
}

with open(SAVE_DIR / 'gcn_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
safe_torch_save({'mean': cont_mean, 'std': cont_std, 'cat_sizes': cat_sizes}, SAVE_DIR / 'gcn_norm.pt')
print('saved:', best_model_path, 'gcn_metadata.json', 'gcn_norm.pt')


device: cpu
graphs: 219 73 74
train positive rate: 0.2206, pos_weight: 3.5326
cat_sizes: {'ORIGIN': 348, 'DEST': 348, 'MONTH': 9, 'DAY_OF_WEEK': 8, 'CRS_DEP_TIME_HOUR': 24, 'CRS_ARR_TIME_HOUR': 24}
Epoch  1 | loss: 0.7399 | val_loss: 0.6592
Epoch  2 | loss: 0.7257 | val_loss: 0.6398
Epoch  3 | loss: 0.7218 | val_loss: 0.6600
Epoch  4 | loss: 0.7193 | val_loss: 0.6790
Epoch  5 | loss: 0.7167 | val_loss: 0.6453
Epoch  6 | loss: 0.7139 | val_loss: 0.6542
Epoch  7 | loss: 0.7128 | val_loss: 0.6467
Epoch  8 | loss: 0.7108 | val_loss: 0.6457
Epoch  9 | loss: 0.7097 | val_loss: 0.7163
Epoch 10 | loss: 0.7092 | val_loss: 0.6969
Early stopping at epoch 10
best thresholds: {'f1': {'threshold': 0.3499999999999999, 'f1': 0.3407648312995293, 'acc': 0.6064594705131205, 'bal_acc': 0.6152612526999659}, 'balanced': {'threshold': 0.32999999999999996, 'f1': 0.34015590715366933, 'acc': 0.5798712494332183, 'bal_acc': 0.6157905476920251}}

[f1] threshold=0.35
ACC: 0.6880
BAL_ACC: 0.5654
AUC: 0.6173
F1:  0.2